# 2026-09-09 데이터 분석 학습 정리
## Chapter 04. pandas로 데이터에 질문하기

---

## 1. 오늘의 학습 주제

오늘은 `pandas`를 이용해 데이터를 단순히 불러오는 것에서 끝나는 것이 아니라, 실제 분석 질문을 데이터 처리 과정으로 바꾸는 방법을 학습했다.

전체적인 분석 흐름은 다음과 같다.

```text
분석 질문 정의
→ 필요한 파일 확인
→ 필요한 컬럼 선택
→ 조건에 맞는 행 필터링
→ 데이터 정렬
→ 파생 컬럼 생성
→ 여러 DataFrame 병합
→ 분석 범위 확정
→ groupby로 그룹화
→ agg로 집계
→ 결과 검증
→ CSV 저장
```

이번 학습에서 가장 중요한 점은 다음과 같다.

> 코드가 오류 없이 실행되었다고 해서 분석 결과까지 올바른 것은 아니다.

데이터의 구조, 컬럼명, 분석 범위, 병합 관계, 행 수, 집계 기준을 반드시 확인해야 한다.

---

# 2. 오늘 사용한 데이터

이번 실습에서는 온라인 쇼핑몰 데이터를 사용했다.

주요 CSV 파일은 다음과 같다.

```text
customers.csv
products.csv
orders.csv
order_items.csv
```

각 파일의 의미는 다음과 같다.

---

## customers

고객 정보를 저장한다.

주요 컬럼 예시:

```text
customer_id
name
gender
age
city
signup_date
```

한 행의 의미:

```text
고객 한 명
```

---

## products

상품 정보를 저장한다.

주요 컬럼 예시:

```text
product_id
product_name
category
price
```

한 행의 의미:

```text
상품 하나
```

---

## orders

주문 정보를 저장한다.

주요 컬럼 예시:

```text
order_id
customer_id
order_date
payment_method
order_status
```

한 행의 의미:

```text
주문 한 건
```

---

## order_items

주문 안에 포함된 상품 상세 정보를 저장한다.

주요 컬럼 예시:

```text
order_item_id
order_id
product_id
quantity
unit_price
```

한 행의 의미:

```text
주문에 포함된 상품 한 항목
```

예를 들어 주문 1개에 상품이 3개 들어 있다면 `orders`에는 1행이 있지만 `order_items`에는 여러 행이 존재할 수 있다.

```text
orders
order_id = 1

order_items
order_id = 1, 상품 A
order_id = 1, 상품 B
order_id = 1, 상품 C
```

---

# 3. CSV 파일 불러오기

`pandas`를 사용해서 CSV 파일을 읽는다.

```python
import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")
```

데이터를 불러온 뒤에는 먼저 구조를 확인해야 한다.

```python
customers.info()
products.info()
orders.info()
order_items.info()
```

---

# 4. info() 사용 시 주의점

다음과 같이 작성할 수도 있다.

```python
print(customers.info())
```

하지만 `info()` 자체가 이미 화면에 결과를 출력하는 함수이기 때문에 마지막에 `None`이 추가로 출력될 수 있다.

따라서 다음처럼 사용하는 것이 좋다.

```python
customers.info()
```

---

# 5. 데이터의 컬럼 확인하기

데이터 분석을 시작하기 전에 실제 컬럼명을 확인하는 습관이 중요하다.

```python
print(customers.columns.tolist())
print(products.columns.tolist())
print(orders.columns.tolist())
print(order_items.columns.tolist())
```

예를 들어 `orders`의 컬럼이 다음과 같다고 가정한다.

```text
[
    'order_id',
    'customer_id',
    'order_date',
    'payment_method',
    'order_status'
]
```

이때 존재하지 않는 컬럼명을 사용하면 `KeyError`가 발생한다.

---

# 6. 오늘 발생한 KeyError

오늘 다음과 같은 오류가 발생했다.

```text
KeyError: 'oreder_status'
```

원인은 컬럼 이름 오타였다.

잘못된 코드:

```python
orders["oreder_status"]
```

올바른 코드:

```python
orders["order_status"]
```

`oreder`가 아니라 `order`가 맞다.

---

또 다른 오타도 있었다.

잘못된 컬럼명:

```text
customers_id
```

올바른 컬럼명:

```text
customer_id
```

따라서 `KeyError`가 발생하면 가장 먼저 다음 코드를 실행한다.

```python
print(orders.columns.tolist())
```

정리하면:

```text
KeyError
→ DataFrame에서 요청한 컬럼을 찾을 수 없음
```

확인할 것:

```text
1. 컬럼명 오타
2. 대소문자
3. 앞뒤 공백
4. 실제 존재하는 컬럼인지
5. merge 후 _x, _y가 붙었는지
```

---

# 7. Series와 DataFrame

pandas에서 컬럼 하나를 선택하면 `Series`가 된다.

```python
customers["city"]
```

결과:

```text
Series
1차원
```

여러 컬럼을 선택하면 `DataFrame`이 된다.

```python
customers[
    ["customer_id", "gender", "age", "city"]
]
```

결과:

```text
DataFrame
2차원
```

차이를 간단하게 정리하면:

```python
customers["city"]
```

```text
Series
```

반면

```python
customers[["city"]]
```

```text
DataFrame
```

이다.

---

# 8. 필요한 컬럼만 선택하기

분석할 때 모든 컬럼을 사용할 필요는 없다.

예를 들어 고객 번호, 나이, 지역만 필요하다면 다음처럼 선택할 수 있다.

```python
customer_view = customers[
    ["customer_id", "age", "city"]
]
```

필요한 컬럼만 사용하는 이유:

```text
1. 데이터를 보기 쉬워진다.
2. 불필요한 정보를 줄일 수 있다.
3. merge 후 중복 컬럼을 줄일 수 있다.
4. 개인정보 노출을 줄일 수 있다.
```

---

# 9. 조건식을 이용한 필터링

특정 조건을 만족하는 행만 선택할 수 있다.

예를 들어 30세 이상 고객을 찾는다면:

```python
customers[
    customers["age"] >= 30
]
```

`customers["age"] >= 30`은 각 행마다 다음과 같은 결과를 만든다.

```text
True
False
True
True
False
```

그리고 `True`인 행만 선택된다.

이것을 `Boolean Mask`, 즉 불리언 마스크라고 한다.

---

# 10. 비교 연산자

pandas에서도 일반적인 비교 연산자를 사용할 수 있다.

```text
==  같다
!=  다르다
>   크다
>=  크거나 같다
<   작다
<=  작거나 같다
```

예:

```python
customers["age"] >= 30
```

```python
customers["city"] == "서울"
```

```python
orders["order_status"] != "cancelled"
```

---

# 11. 여러 조건 사용하기

pandas에서는 여러 조건을 사용할 때 Python의 `and`, `or`, `not` 대신 다음 기호를 사용한다.

```text
&   AND
|   OR
~   NOT
```

---

## AND 조건

30세 이상이면서 서울에 거주하는 고객:

```python
customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]
```

조건을 여러 개 사용할 때 각각 괄호로 묶는 것이 중요하다.

---

## OR 조건

서울 또는 부산에 거주하는 고객:

```python
customers[
    (customers["city"] == "서울")
    | (customers["city"] == "부산")
]
```

---

## NOT 조건

완료 주문이 아닌 주문:

```python
orders[
    ~(orders["order_status"] == "completed")
]
```

---

# 12. isin()

여러 값 중 하나에 해당하는 데이터를 찾을 때 `isin()`을 사용한다.

예를 들어 서울 또는 부산 고객:

```python
customers[
    customers["city"].isin(["서울", "부산"])
]
```

다음 조건과 비슷한 의미다.

```python
(customers["city"] == "서울")
| (customers["city"] == "부산")
```

특정 값을 제외할 수도 있다.

```python
orders[
    ~orders["order_status"].isin(
        ["cancelled", "refunded"]
    )
]
```

---

# 13. value_counts()

컬럼 안에 어떤 값이 얼마나 들어 있는지 확인할 때 사용한다.

예:

```python
orders["order_status"].value_counts()
```

결과 예시:

```text
completed    100
cancelled     20
refunded      10
```

결측치까지 확인하려면:

```python
orders["order_status"].value_counts(
    dropna=False
)
```

비율을 보고 싶다면:

```python
orders["payment_method"].value_counts(
    normalize=True
)
```

백분율로 바꾸려면:

```python
orders["payment_method"].value_counts(
    normalize=True,
    dropna=False
).mul(100).round(1)
```

---

# 14. sort_values()

데이터를 원하는 기준으로 정렬할 수 있다.

가격이 높은 순서:

```python
products.sort_values(
    "price",
    ascending=False
)
```

`ascending=False`

```text
내림차순
큰 값 → 작은 값
```

`ascending=True`

```text
오름차순
작은 값 → 큰 값
```

가격이 높은 상품 상위 10개:

```python
expensive_products = (
    products
    .sort_values(
        "price",
        ascending=False
    )
    .head(10)
)
```

---

# 15. 여러 기준으로 정렬하기

카테고리는 오름차순, 가격은 내림차순으로 정렬할 수도 있다.

```python
products.sort_values(
    ["category", "price"],
    ascending=[True, False]
)
```

---

# 16. copy()

필터링한 데이터를 별도로 수정할 예정이라면 `.copy()`를 사용하는 것이 좋다.

```python
completed_orders = orders[
    orders["order_status"] == "completed"
].copy()
```

장점:

```text
1. 원본과 작업 데이터를 구분할 수 있다.
2. 의도하지 않은 원본 변경을 줄일 수 있다.
3. SettingWithCopyWarning 가능성을 줄일 수 있다.
```

---

# 17. 파생 컬럼 만들기

기존 컬럼을 계산해서 새로운 컬럼을 만들 수 있다.

이번 실습에서는 다음 값을 만들었다.

```text
line_total
```

계산식:

```text
line_total = quantity × unit_price
```

코드:

```python
order_items_work = order_items.copy()

order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)
```

예:

```text
quantity = 2
unit_price = 10,000
```

그러면:

```text
line_total = 20,000
```

---

# 18. 파생 컬럼 검증하기

파생 컬럼을 만들었다면 일부 데이터를 직접 계산해 확인하는 것이 좋다.

```python
sample = order_items_work.iloc[0]

expected = (
    sample["quantity"]
    * sample["unit_price"]
)

actual = sample["line_total"]

print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)
```

결과가 `True`라면 계산 결과가 일치한다.

---

# 19. line_total의 의미

중요한 점은 `line_total`이 주문 전체 금액이 아니라는 것이다.

```text
line_total
= 주문상세 한 행의 금액
```

즉:

```text
상품 한 항목의 수량 × 판매 단가
```

이다.

하나의 주문에 상품이 여러 개 있다면 하나의 주문에는 여러 개의 `line_total`이 존재할 수 있다.

---

# 20. 전체 주문상세 금액

전체 주문상세 금액은 다음처럼 계산할 수 있다.

```python
all_order_amount = (
    order_items_work["line_total"].sum()
)

print(all_order_amount)
```

하지만 이 값에는 다음 주문이 모두 포함될 수 있다.

```text
completed
cancelled
refunded
```

따라서 주문 상태를 확인하기 전에는 이 값을 바로 `매출`이라고 부르면 안 된다.

더 정확한 표현은:

```text
전체 주문상세 금액
```

이다.

---

# 21. 완료 주문 기준 매출

실제 매출을 분석하려면 주문 상태가 `completed`인 주문만 선택해야 한다.

```text
order_status == "completed"
```

즉:

```text
전체 주문상세 금액
≠
완료 주문 매출
```

완료 주문을 필터링한 뒤 `line_total`을 합해야 완료 주문 기준 매출이라고 할 수 있다.

---

# 22. groupby()

`groupby()`는 데이터를 특정 기준으로 그룹화하는 기능이다.

예를 들어 상품 카테고리별 평균 가격:

```python
products.groupby(
    "category"
)["price"].mean()
```

`groupby()`에서 가장 중요한 개념은 다음과 같다.

> groupby의 기준 컬럼이 결과 표에서 한 행이 무엇을 의미하는지 결정한다.

예:

```python
groupby("category")
```

결과 한 행:

```text
카테고리 하나
```

```python
groupby("customer_id")
```

결과 한 행:

```text
고객 한 명
```

```python
groupby("order_month")
```

결과 한 행:

```text
한 달
```

---

# 23. agg()

`agg()`는 그룹별로 여러 가지 계산을 동시에 수행할 때 사용한다.

예:

```python
category_price_summary = (
    products
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        product_count=(
            "product_id",
            "nunique"
        ),
        average_price=(
            "price",
            "mean"
        ),
        minimum_price=(
            "price",
            "min"
        ),
        maximum_price=(
            "price",
            "max"
        ),
    )
)
```

각 결과의 의미:

```text
product_count
→ 상품 종류 수

average_price
→ 평균 가격

minimum_price
→ 최소 가격

maximum_price
→ 최대 가격
```

---

# 24. agg의 의미

`agg`는 `aggregate`의 줄임말이다.

쉽게 말하면:

```text
여러 데이터를 모아서
대표적인 값으로 요약한다.
```

예를 들어:

```text
합계
평균
최솟값
최댓값
개수
고유값 개수
```

등을 계산할 수 있다.

---

# 25. as_index=False

`groupby()`에서 다음처럼 사용할 수 있다.

```python
.groupby(
    "category",
    as_index=False
)
```

`as_index=False`를 사용하면 그룹 기준인 `category`가 일반 컬럼으로 유지된다.

따라서 이후:

```text
저장
병합
정렬
출력
```

등을 하기 편하다.

---

# 26. count(), size(), nunique() 차이

세 함수의 차이를 알아야 한다.

---

## count()

결측값을 제외한 값의 개수를 센다.

```python
df["column"].count()
```

---

## size()

그룹 안의 전체 행 수를 센다.

```python
group.size()
```

---

## nunique()

중복을 제외한 고유 값의 개수를 센다.

```python
orders["order_id"].nunique()
```

---

# 27. 주문상세 행 수와 실제 주문 수

`order_items`는 주문 하나에 여러 상품이 있을 수 있기 때문에 행 수와 주문 수가 다를 수 있다.

```python
len(order_items)
```

이 값은:

```text
주문상세 행 수
```

이다.

실제 주문 건수는:

```python
order_items["order_id"].nunique()
```

로 계산한다.

즉:

```text
len(order_items)
≠ 실제 주문 수
```

---

# 28. merge()

서로 다른 DataFrame에 있는 정보를 연결할 때 `merge()`를 사용한다.

예를 들어 `order_items`에는 다음 정보가 있다.

```text
order_id
product_id
quantity
unit_price
```

하지만 다음 정보는 없다.

```text
customer_id
order_date
order_status
```

이 정보는 `orders`에 있기 때문에 두 데이터를 연결해야 한다.

```python
order_items.merge(
    orders,
    on="order_id",
    how="left"
)
```

---

# 29. merge의 연결 키

두 DataFrame을 연결할 때 공통 기준이 필요하다.

예:

```text
order_items.order_id
        ↓
orders.order_id
```

코드에서는:

```python
on="order_id"
```

라고 작성한다.

---

# 30. merge 방식

대표적인 병합 방식은 다음과 같다.

---

## left

왼쪽 데이터를 모두 유지한다.

```python
how="left"
```

---

## right

오른쪽 데이터를 모두 유지한다.

```python
how="right"
```

---

## inner

양쪽에 모두 존재하는 키만 유지한다.

```python
how="inner"
```

---

## outer

양쪽의 모든 키를 유지한다.

```python
how="outer"
```

---

# 31. 데이터 관계

병합을 할 때 데이터 관계를 이해해야 한다.

대표적인 관계:

```text
one_to_one
one_to_many
many_to_one
many_to_many
```

한국어로:

```text
일대일
일대다
다대일
다대다
```

---

# 32. many_to_one

오늘 특히 중요하게 배운 관계가 `many_to_one`이다.

예를 들어 `order_items`에는 같은 주문 번호가 여러 번 존재할 수 있다.

```text
order_items

order_id
1
1
1
2
2
3
```

왜냐하면 주문 하나에 여러 상품이 들어갈 수 있기 때문이다.

반면 `orders`에서는 주문 한 건당 한 행이 존재한다.

```text
orders

order_id
1
2
3
```

따라서:

```text
order_items → orders
```

관계는:

```text
many_to_one
```

이다.

쉽게 표현하면:

```text
주문상세 여러 개
→ 주문 하나
```

이다.

---

# 33. 대표적인 many_to_one 관계

이번 데이터에서는 다음 관계가 대표적인 `many_to_one`이다.

```text
order_items.order_id
→ orders.order_id
```

그리고:

```text
order_items.product_id
→ products.product_id
```

도 `many_to_one` 관계다.

상품 하나는 여러 주문상세에 반복해서 등장할 수 있지만 `products`에서는 상품 하나당 한 행이 존재하기 때문이다.

---

# 34. validate

`merge()`를 사용할 때 예상한 관계가 맞는지 검증할 수 있다.

```python
order_sales = order_items_work.merge(
    orders_for_merge,
    on="order_id",
    how="left",
    validate="many_to_one"
)
```

`validate="many_to_one"`의 의미:

```text
왼쪽 키
→ 여러 번 등장 가능

오른쪽 키
→ 반드시 고유해야 함
```

오른쪽 테이블의 키가 중복되어 있다면 pandas가 오류를 발생시킨다.

이를 통해 잘못된 병합을 조기에 발견할 수 있다.

---

# 35. 오늘 직접 수정한 merge 코드

오늘 작성한 코드에서 컬럼명 오타를 수정했다.

잘못된 코드:

```python
order_sales = order_items.merge(
    orders[
        [
            "order_id",
            "customers_id",
            "order_date",
            "order_status"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)
```

`customers_id`가 잘못된 컬럼명이었다.

수정:

```python
order_sales = order_items.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

print(order_sales.head())
```

---

# 36. 병합용 컬럼만 선택하기

merge를 할 때 전체 컬럼을 다 가져오는 것보다 필요한 컬럼만 선택하는 것이 좋다.

```python
orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()
```

이렇게 하면 불필요한 컬럼이 추가되는 것을 줄일 수 있다.

---

# 37. 안전하게 주문 데이터 병합하기

```python
order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)
```

여기서는:

```text
on
→ 어떤 컬럼을 기준으로 연결할 것인가

how
→ 어떤 방식으로 연결할 것인가

validate
→ 데이터 관계가 예상과 맞는가

indicator
→ 실제 연결 결과가 어떻게 되었는가
```

를 설정한다.

---

# 38. indicator

`indicator`를 사용하면 병합된 데이터가 어디에서 왔는지 확인할 수 있다.

```python
indicator="order_match"
```

그러면 `order_match`라는 컬럼이 생성된다.

확인:

```python
order_sales[
    "order_match"
].value_counts(
    dropna=False
)
```

대표적인 결과:

```text
both
left_only
right_only
```

---

## both

양쪽 데이터에서 키를 찾았다는 뜻이다.

```text
정상적으로 연결됨
```

---

## left_only

왼쪽에는 데이터가 있지만 오른쪽에서 연결할 데이터를 찾지 못했다는 뜻이다.

예:

```text
order_items에는 order_id가 있는데
orders에는 같은 order_id가 없음
```

---

## right_only

오른쪽에는 있지만 왼쪽에는 없는 데이터다.

단, `left merge`에서는 일반적으로 `right_only`가 나타나지 않는다.

---

# 39. 병합 전후 행 수 확인

병합 후에는 행 수를 반드시 확인해야 한다.

```python
print(
    "병합 전 행 수:",
    len(order_items_work)
)

print(
    "병합 후 행 수:",
    len(order_sales)
)
```

정상적인 `many_to_one` 병합이라면 일반적으로 행 수가 유지된다.

예:

```text
병합 전: 1000
병합 후: 1000
```

만약:

```text
병합 전: 1000
병합 후: 1500
```

처럼 증가한다면 문제가 있을 수 있다.

확인할 사항:

```text
1. 오른쪽 키 중복
2. 잘못된 병합 키
3. many_to_many 관계 발생
4. 잘못된 데이터 구조
```

---

# 40. 중복 키 확인하기

예를 들어 `orders.order_id`가 고유한지 확인한다.

```python
orders[
    "order_id"
].duplicated().sum()
```

정상이라면:

```text
0
```

이 나오는 것이 좋다.

중복 데이터를 직접 확인하려면:

```python
orders[
    orders[
        "order_id"
    ].duplicated(
        keep=False
    )
].sort_values(
    "order_id"
)
```

---

# 41. _x, _y 컬럼

merge를 했을 때 양쪽 DataFrame에 같은 이름의 컬럼이 있으면 pandas가 자동으로 `_x`, `_y`를 붙일 수 있다.

예:

```text
price_x
price_y
```

확인:

```python
[
    col
    for col in merged.columns
    if col.endswith("_x")
    or col.endswith("_y")
]
```

이를 줄이기 위해 병합하기 전에 필요한 컬럼만 선택하는 것이 좋다.

---

# 42. 완료 주문 분석셋 만들기

주문 상태를 연결한 뒤 `completed` 주문만 선택한다.

```python
completed_sales = order_sales[
    order_sales[
        "order_status"
    ] == "completed"
].copy()
```

이제 이 데이터를 이용해서 실제 완료 주문 기준 매출을 계산할 수 있다.

---

# 43. 완료 주문 매출

```python
completed_revenue = (
    completed_sales[
        "line_total"
    ].sum()
)
```

---

# 44. 완료 주문 수

주문 수는 주문상세 행 개수가 아니라 고유 `order_id` 개수로 계산한다.

```python
completed_order_count = (
    completed_sales[
        "order_id"
    ].nunique()
)
```

---

# 45. 완료 주문 고객 수

```python
completed_customer_count = (
    completed_sales[
        "customer_id"
    ].nunique()
)
```

---

# 46. 완료 주문 분석 기준

이번 분석에서는 다음과 같이 기준을 정의할 수 있다.

```text
매출 기준
→ completed 주문

금액 기준
→ quantity × unit_price

주문 수
→ 고유 order_id 수

고객 수
→ 고유 customer_id 수

cancelled
→ 매출에서 제외

refunded
→ 매출에서 제외
```

---

# 47. 상품 데이터 병합

완료 주문상세 데이터에 상품명과 카테고리를 연결한다.

먼저 필요한 상품 컬럼만 선택한다.

```python
products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()
```

그리고 merge한다.

```python
completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)
```

---

# 48. 상품 가격과 실제 판매 단가 구분

`products`에는 상품의 `price`가 있을 수 있다.

하지만 실제 주문 당시 판매 단가는 `order_items`의:

```text
unit_price
```

이다.

따라서 매출을 계산할 때 상품 마스터의 `price`를 사용하는 것이 아니라:

```text
quantity × unit_price
```

를 사용해야 한다.

---

# 49. 카테고리별 매출

상품 정보를 연결하면 카테고리별 매출을 계산할 수 있다.

```python
category_sales = (
    completed_items
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        total_sales=(
            "line_total",
            "sum"
        ),
        order_count=(
            "order_id",
            "nunique"
        ),
        customer_count=(
            "customer_id",
            "nunique"
        ),
        quantity_sold=(
            "quantity",
            "sum"
        ),
        detail_row_count=(
            "order_item_id",
            "count"
        ),
    )
    .sort_values(
        "total_sales",
        ascending=False
    )
)
```

결과에서 한 행은:

```text
카테고리 하나
```

를 의미한다.

---

# 50. 카테고리별 결과 컬럼 의미

```text
category
→ 카테고리

total_sales
→ 완료 주문 매출 합계

order_count
→ 고유 주문 수

customer_count
→ 고유 고객 수

quantity_sold
→ 판매 수량 합계

detail_row_count
→ 주문상세 행 수
```

---

# 51. 카테고리 합계 검증

카테고리별 매출을 모두 합한 값과 전체 완료 주문 매출이 같은지 확인한다.

```python
category_total = (
    category_sales[
        "total_sales"
    ].sum()
)

completed_total = (
    completed_items[
        "line_total"
    ].sum()
)

print(category_total)
print(completed_total)

print(
    category_total
    == completed_total
)
```

결과가:

```text
True
```

라면 두 합계가 일치한다.

---

# 52. 합계가 다를 때 확인할 것

```text
1. category 결측치
2. product_id 미매칭
3. 중복 병합
4. 필터 조건 차이
5. 완료 주문 범위 차이
```

---

# 53. 상품별 매출

```python
product_sales = (
    completed_items
    .groupby(
        [
            "product_id",
            "product_name",
            "category"
        ],
        as_index=False,
    )
    .agg(
        total_sales=(
            "line_total",
            "sum"
        ),
        quantity_sold=(
            "quantity",
            "sum"
        ),
        order_count=(
            "order_id",
            "nunique"
        ),
        customer_count=(
            "customer_id",
            "nunique"
        ),
    )
    .sort_values(
        "total_sales",
        ascending=False
    )
)
```

결과 한 행은:

```text
상품 하나
```

를 의미한다.

---

# 54. 판매량과 매출은 다를 수 있다

판매량이 높은 상품과 매출이 높은 상품은 반드시 같지 않다.

예를 들어:

```text
상품 A
판매량 100개
가격 1,000원
매출 100,000원

상품 B
판매량 20개
가격 100,000원
매출 2,000,000원
```

따라서 `인기 상품`이라는 표현을 사용할 때도 기준을 명확하게 해야 한다.

```text
판매량 기준인지
매출 기준인지
주문 수 기준인지
```

를 구분해야 한다.

---

# 55. 날짜 데이터 변환

날짜 분석을 하기 위해 문자열 데이터를 날짜 타입으로 변환한다.

```python
completed_items[
    "order_date"
] = pd.to_datetime(
    completed_items[
        "order_date"
    ],
    errors="coerce"
)
```

`errors="coerce"`를 사용하면 변환에 실패한 값은:

```text
NaT
```

로 처리된다.

---

# 56. 날짜 변환 실패 확인

```python
print(
    completed_items[
        "order_date"
    ].isna().sum()
)
```

이를 통해 날짜 변환에 실패한 데이터가 몇 개인지 확인할 수 있다.

---

# 57. 월 컬럼 만들기

날짜에서 월 정보를 추출한다.

```python
completed_items[
    "order_month"
] = (
    completed_items[
        "order_date"
    ]
    .dt.to_period("M")
    .astype("string")
)
```

예:

```text
2026-01
2026-02
2026-03
```

---

# 58. 월별 매출

```python
monthly_sales = (
    completed_items
    .groupby(
        "order_month",
        as_index=False
    )
    .agg(
        total_sales=(
            "line_total",
            "sum"
        ),
        order_count=(
            "order_id",
            "nunique"
        ),
        customer_count=(
            "customer_id",
            "nunique"
        ),
        quantity_sold=(
            "quantity",
            "sum"
        ),
    )
    .sort_values(
        "order_month"
    )
)
```

결과 한 행은:

```text
한 달
```

을 의미한다.

---

# 59. 고객별 구매 금액

```python
customer_sales = (
    completed_items
    .groupby(
        "customer_id",
        as_index=False
    )
    .agg(
        total_sales=(
            "line_total",
            "sum"
        ),
        order_count=(
            "order_id",
            "nunique"
        ),
        quantity_sold=(
            "quantity",
            "sum"
        ),
    )
)
```

결과 한 행은:

```text
고객 한 명
```

을 의미한다.

---

# 60. 고객 속성 연결

고객별 집계 결과에 고객 정보를 연결할 수도 있다.

개인정보를 최소화하기 위해 필요한 컬럼만 선택한다.

```python
customer_attributes = customers[
    [
        "customer_id",
        "gender",
        "age",
        "city"
    ]
].copy()
```

병합:

```python
customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values(
        "total_sales",
        ascending=False
    )
)
```

---

# 61. one_to_one

고객별로 이미 집계한 `customer_sales`에는 고객 한 명당 한 행이 있다.

`customers` 역시 고객 한 명당 한 행이 존재한다.

따라서:

```text
customer_sales.customer_id
→ customers.customer_id
```

관계는:

```text
one_to_one
```

이다.

---

# 62. 결과 저장 폴더 만들기

```python
output_dir = (
    project_root
    / "reports"
    / "chapter04"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)
```

---

# 63. CSV 저장하기

카테고리별 매출 저장:

```python
category_sales.to_csv(
    output_dir
    / "category_sales.csv",
    index=False,
    encoding="utf-8-sig",
)
```

---

# 64. index=False

```python
index=False
```

의 의미:

```text
pandas의 인덱스를
CSV 파일에 별도 컬럼으로 저장하지 않는다.
```

---

# 65. utf-8-sig

```python
encoding="utf-8-sig"
```

를 사용하면 Windows Excel 등에서 한글이 깨지는 문제를 줄일 수 있다.

---

# 66. 여러 결과 파일 저장

```python
outputs = {
    "category_sales.csv":
        category_sales,

    "product_sales.csv":
        product_sales,

    "monthly_sales.csv":
        monthly_sales,

    "customer_sales.csv":
        customer_sales_detail,
}

for file_name, df in outputs.items():

    output_path = (
        output_dir / file_name
    )

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
```

---

# 67. 저장 결과 확인

파일이 실제로 생성됐는지 확인할 수 있다.

```python
print(
    output_path.exists()
)
```

파일 크기도 확인할 수 있다.

```python
print(
    output_path.stat().st_size
)
```

---

# 68. 저장한 CSV 다시 읽기

저장이 끝났다고 바로 끝내지 않고 다시 읽어 확인하는 것이 좋다.

```python
saved_category_sales = (
    pd.read_csv(
        output_dir
        / "category_sales.csv"
    )
)

print(
    saved_category_sales.head()
)
```

확인할 것:

```text
1. 컬럼
2. 행 수
3. 한글 표시
4. 값
5. 파일 저장 여부
```

---

# 69. 오늘 발생한 ModuleNotFoundError

오늘 실습 중 다음 오류가 발생했다.

```text
ModuleNotFoundError:
No module named 'course_utils'
```

처음에는 CSV 파일 경로 문제처럼 보였지만 실제로는 Python이 `course_utils` 모듈을 찾지 못한 것이 원인이었다.

프로젝트 구조는 다음과 비슷했다.

```text
llm_data_analysis-course
│
├─ course_utils
│  ├─ __init__.py
│  └─ path.py
│
├─ data
│  └─ raw
│     ├─ customers.csv
│     ├─ products.csv
│     ├─ orders.csv
│     └─ order_items.csv
│
└─ notebooks
   └─ ch04_sub
      └─ 00.ipynb
```

현재 Notebook 위치가:

```text
notebooks/ch04_sub
```

였기 때문에 프로젝트 루트에 있는 `course_utils`를 Python이 바로 찾지 못할 수 있었다.

---

# 70. 현재 실행 위치 확인

```python
from pathlib import Path

print(
    Path.cwd()
)
```

예:

```text
C:\dev\llm_data_analysis-course
\notebooks\ch04_sub
```

---

# 71. 프로젝트 루트를 Python 경로에 추가

```python
import sys

from pathlib import Path

PROJECT_ROOT = (
    Path.cwd()
    .parent
    .parent
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )

print(
    "프로젝트 루트:",
    PROJECT_ROOT
)
```

그러면:

```python
from course_utils.path import (
    get_project_root,
    get_data_dir
)
```

를 사용할 수 있다.

---

# 72. Path 객체로 파일 경로 연결

`Path` 객체를 사용할 때는 `/` 연산자를 이용해서 폴더와 파일을 연결할 수 있다.

```python
get_data_dir() / "raw" / "customers.csv"
```

예:

```text
C:\dev\llm_data_analysis-course
\data
\raw
\customers.csv
```

코드:

```python
customers = pd.read_csv(
    get_data_dir()
    / "raw"
    / "customers.csv"
)
```

---

# 73. 파일 존재 여부 확인

파일을 읽기 전에 실제 파일이 존재하는지 확인할 수 있다.

```python
customers_path = (
    get_data_dir()
    / "raw"
    / "customers.csv"
)

print(
    customers_path
)

print(
    customers_path.exists()
)
```

결과:

```text
True
```

라면 파일이 존재한다는 뜻이다.

---

# 74. 오류를 볼 때 핵심 메시지 확인하기

Python 오류는 굉장히 길게 출력될 수 있다.

하지만 가장 아래쪽에 있는 오류 이름과 메시지를 먼저 보는 것이 중요하다.

예:

```text
KeyError: 'oreder_status'
```

핵심은:

```text
KeyError
```

와:

```text
oreder_status
```

이다.

즉:

```text
oreder_status라는 컬럼을 찾지 못했다.
```

라는 뜻이다.

---

# 75. pandas 오류 확인 순서

오류가 발생하면 다음 순서로 확인하는 습관이 좋다.

---

## 1단계. DataFrame 확인

```python
print(
    type(orders)
)
```

---

## 2단계. 행과 열 개수 확인

```python
print(
    orders.shape
)
```

---

## 3단계. 컬럼 확인

```python
print(
    orders.columns.tolist()
)
```

---

## 4단계. 데이터 일부 확인

```python
print(
    orders.head()
)
```

---

## 5단계. 실제 값 확인

```python
print(
    orders[
        "order_status"
    ].value_counts(
        dropna=False
    )
)
```

---

## 6단계. 병합 키 확인

```python
print(
    orders[
        "order_id"
    ].duplicated().sum()
)
```

---

## 7단계. 병합 전후 행 수 확인

```python
print(
    len(order_items_work)
)

print(
    len(order_sales)
)
```

---

## 8단계. 미매칭 확인

```python
print(
    order_sales[
        "order_match"
    ].value_counts(
        dropna=False
    )
)
```

---

# 76. KeyError 해결 방법

`KeyError`가 발생하면:

```python
print(
    df.columns.tolist()
)
```

추가로 공백까지 확인하고 싶다면:

```python
print(
    [
        repr(col)
        for col
        in df.columns
    ]
)
```

확인:

```text
컬럼명
대소문자
앞뒤 공백
_x / _y
오타
```

---

# 77. pandas에서 and를 사용하면 발생할 수 있는 오류

잘못된 코드:

```python
customers[
    (customers["age"] >= 30)
    and
    (customers["city"] == "서울")
]
```

pandas에서는 `and` 대신 `&`를 사용한다.

수정:

```python
customers[
    (customers["age"] >= 30)
    &
    (customers["city"] == "서울")
]
```

---

# 78. pandas 조건 연산자 정리

```text
Python     pandas

and        &
or         |
not        ~
```

그리고 각 조건은 괄호로 묶는다.

```python
(
    customers["age"] >= 30
)
&
(
    customers["city"] == "서울"
)
```

---

# 79. 필터 결과가 비어 있을 때

필터 결과가 0행이라고 해서 코드 오류라고 단정하면 안 된다.

먼저 실제 값을 확인한다.

```python
customers[
    "city"
].value_counts(
    dropna=False
)
```

확인할 것:

```text
1. 실제 값 표기
2. 한글/영문 차이
3. 대소문자
4. 공백
5. 결측치
6. 조건 범위
7. 컬럼명
```

---

# 80. MergeError

다음과 같이 선언했는데:

```python
validate="many_to_one"
```

오류가 발생했다면 오른쪽 데이터의 기준 키가 고유하지 않을 가능성이 있다.

예:

```python
orders[
    "order_id"
].duplicated().sum()
```

중복이 있다면 `many_to_one` 관계가 깨질 수 있다.

---

# 81. 병합 후 행 수가 증가할 때

병합 전:

```text
1000행
```

병합 후:

```text
1500행
```

이라면 확인해야 한다.

가능한 원인:

```text
오른쪽 키 중복

잘못된 merge 키

many_to_many 관계

병합 전에 집계해야 하는 데이터를
원본 그대로 연결함
```

---

# 82. SettingWithCopyWarning

필터링한 DataFrame에 새로운 컬럼을 추가할 때 경고가 발생할 수 있다.

따라서:

```python
filtered = (
    df[condition]
    .copy()
)
```

처럼 복사본을 만든다.

그 후:

```python
filtered[
    "new_column"
] = ...
```

처럼 수정한다.

---

# 83. 날짜 처리 오류

먼저 데이터 타입을 확인한다.

```python
print(
    orders[
        "order_date"
    ].dtype
)
```

필요하면 날짜 타입으로 변환한다.

```python
orders_work = (
    orders.copy()
)

orders_work[
    "order_date"
] = pd.to_datetime(
    orders_work[
        "order_date"
    ],
    errors="coerce"
)
```

변환 실패 확인:

```python
print(
    orders_work[
        "order_date"
    ].isna().sum()
)
```

---

# 84. 분석에서 중요한 질문

코드를 작성하기 전에 다음 질문을 먼저 생각해야 한다.

```text
이 데이터에서 한 행은 무엇인가?

어떤 DataFrame이 필요한가?

어떤 컬럼이 필요한가?

어떤 조건의 행만 사용할 것인가?

매출의 기준은 무엇인가?

주문 수의 기준은 무엇인가?

어떤 컬럼을 기준으로 groupby 할 것인가?

어떤 키로 merge 할 것인가?

데이터 관계는 무엇인가?

병합 후 행 수가 유지되는가?

미매칭 데이터가 있는가?

집계 합계가 원본 합계와 일치하는가?
```

---

# 85. 매출의 정의

이번 분석에서 매출은 다음과 같이 정의했다.

```text
order_status가 completed인 주문
+
quantity × unit_price
```

즉:

```text
매출
=
completed 주문의
line_total 합계
```

---

# 86. 주문 수의 정의

주문 수는 단순한 행 개수가 아니다.

```python
completed_sales[
    "order_id"
].nunique()
```

를 사용한다.

즉:

```text
주문 수
=
고유 order_id 개수
```

---

# 87. 고객 수의 정의

```python
completed_sales[
    "customer_id"
].nunique()
```

즉:

```text
고객 수
=
고유 customer_id 개수
```

---

# 88. 판매량의 정의

```python
completed_sales[
    "quantity"
].sum()
```

즉:

```text
판매량
=
quantity 합계
```

---

# 89. 분석 단위를 명확하게 해야 하는 이유

같은 데이터라도 어떤 기준으로 묶느냐에 따라 결과의 의미가 달라진다.

```text
groupby("category")
→ 카테고리별 분석

groupby("product_id")
→ 상품별 분석

groupby("customer_id")
→ 고객별 분석

groupby("order_month")
→ 월별 분석

groupby("order_status")
→ 주문 상태별 분석
```

따라서 `groupby()`는 단순한 pandas 문법이 아니라 결과의 분석 단위를 결정하는 중요한 기능이다.

---

# 90. 오늘 배운 핵심 함수

```python
pd.read_csv()

df.info()

df.head()

df.shape

df.columns.tolist()

df.copy()

df.isin()

df.value_counts()

df.sort_values()

df.groupby()

df.agg()

df.count()

df.size()

df.nunique()

df.merge()

df.duplicated()

pd.to_datetime()

df.to_csv()
```

---

# 91. 오늘 배운 핵심 개념 정리

```text
Series
= 1차원 데이터

DataFrame
= 2차원 표 데이터

Boolean Mask
= True / False 조건으로 행 선택

&
= AND

|
= OR

~
= NOT

isin()
= 여러 값 중 하나에 해당하는지 확인

sort_values()
= 정렬

copy()
= 작업용 데이터 복사

line_total
= quantity × unit_price

value_counts()
= 값의 빈도 확인

groupby()
= 데이터를 기준별로 그룹화

agg()
= 그룹별 여러 값을 집계

count()
= 결측값 제외 개수

size()
= 전체 행 수

nunique()
= 고유 값 개수

merge()
= 여러 DataFrame 연결

one_to_one
= 일대일

one_to_many
= 일대다

many_to_one
= 다대일

many_to_many
= 다대다

validate
= merge 관계 검증

indicator
= merge 연결 결과 확인

KeyError
= 요청한 컬럼을 찾지 못함

ModuleNotFoundError
= Python이 모듈을 찾지 못함
```

---

# 92. 오늘 가장 중요하게 배운 many_to_one

다시 정리하면:

```text
order_items
```

에는 주문 하나당 여러 상품이 존재할 수 있다.

따라서 같은 `order_id`가 여러 번 존재할 수 있다.

반면:

```text
orders
```

에서는 하나의 `order_id`가 한 행만 존재해야 한다.

그래서:

```text
order_items
        ↓
orders
```

관계는:

```text
many_to_one
```

이다.

쉽게 표현하면:

```text
여러 주문상세
→ 하나의 주문
```

이다.

---

# 93. 오늘 가장 중요하게 배운 agg

`agg()`는 그룹별로 여러 요약 값을 한 번에 계산한다.

예:

```python
.groupby(
    "category",
    as_index=False
)
.agg(
    total_sales=(
        "line_total",
        "sum"
    ),
    order_count=(
        "order_id",
        "nunique"
    ),
    customer_count=(
        "customer_id",
        "nunique"
    ),
)
```

쉽게 표현하면:

```text
category별로 묶고

매출 합계도 계산하고
주문 수도 계산하고
고객 수도 계산한다.
```

---

# 94. 오늘 가장 중요하게 배운 검증

데이터 분석에서는 단순히 결과를 만드는 것보다 결과를 검증하는 것이 중요하다.

병합 후에는:

```text
행 수가 유지되는가?

미매칭 데이터가 있는가?

키가 중복되어 있지 않은가?

예상한 관계가 맞는가?
```

를 확인한다.

집계 후에는:

```text
원본 합계와
집계 합계가 일치하는가?
```

를 확인한다.

---

# 95. LLM이 작성한 코드도 검증해야 한다

AI가 코드를 만들어 주더라도 그대로 사용하는 것이 아니라 다음을 확인해야 한다.

```text
1. 실제 DataFrame 이름인가?

2. 실제 존재하는 컬럼인가?

3. order_status 값이 실제로 completed인가?

4. line_total 계산식이 맞는가?

5. 분석 범위가 완료 주문인가?

6. 주문 수에 nunique()를 사용하는가?

7. merge 키가 올바른가?

8. many_to_one 관계가 맞는가?

9. validate를 사용했는가?

10. indicator로 미매칭을 확인했는가?

11. 병합 전후 행 수가 유지되는가?

12. 최종 합계와 원본 합계가 일치하는가?
```

---

# 96. 실행 성공과 분석 성공은 다르다

코드가 정상적으로 실행되더라도 다음과 같은 문제가 있을 수 있다.

```text
취소 주문까지 매출에 포함

환불 주문까지 포함

주문상세 행 수를 주문 수로 계산

상품 마스터 가격을
실제 판매 가격으로 사용

잘못된 merge로 행 수 증가

미매칭 데이터 누락

날짜를 문자열 상태로 분석

잘못된 groupby 기준 사용

결과를 과도하게 해석
```

따라서:

> 코드 실행 성공 ≠ 분석 결과의 정확성

이다.

---

# 97. 오늘 실제로 경험한 문제 정리

## 문제 1

```text
ModuleNotFoundError:
No module named 'course_utils'
```

원인:

```text
Notebook 실행 위치와
Python 모듈 검색 경로 문제
```

해결:

```text
현재 경로 확인

프로젝트 루트 확인

필요하면 sys.path에
프로젝트 루트 추가
```

---

## 문제 2

```text
KeyError:
'oreder_status'
```

원인:

```text
컬럼 이름 오타
```

수정:

```text
oreder_status
→ order_status
```

---

## 문제 3

```text
customers_id
```

실제 컬럼:

```text
customer_id
```

따라서 실제 컬럼명 확인이 중요하다.

```python
orders.columns.tolist()
```

---

# 98. 오늘 배운 오류 해결 습관

오류가 발생했을 때 바로 전체 코드를 바꾸지 않는다.

먼저 작은 단위로 확인한다.

```text
1. 현재 위치 확인

2. 파일 경로 확인

3. 파일 존재 여부 확인

4. DataFrame 존재 여부 확인

5. columns 확인

6. head() 확인

7. 실제 범주값 확인

8. 키 중복 확인

9. merge 관계 확인

10. 병합 전후 행 수 확인
```

---

# 99. 오늘의 전체 분석 흐름 다시 정리

```text
CSV 불러오기
↓
데이터 구조 확인
↓
컬럼명 확인
↓
필요한 컬럼 선택
↓
조건 필터링
↓
데이터 정렬
↓
작업용 복사본 생성
↓
line_total 생성
↓
수작업 계산 검증
↓
orders와 merge
↓
many_to_one 검증
↓
indicator 확인
↓
병합 전후 행 수 확인
↓
completed 주문 필터링
↓
products와 merge
↓
카테고리별 groupby
↓
agg로 매출·주문수·고객수 집계
↓
상품별 집계
↓
날짜 변환
↓
월별 집계
↓
고객별 집계
↓
원본 합계와 집계 합계 비교
↓
CSV 저장
↓
다시 읽어서 검증
```

---

# 100. 오늘 학습 한 줄 요약

> pandas 데이터 분석은 단순히 함수를 외우는 것이 아니라 분석 질문에 맞는 데이터를 선택하고, 조건에 맞게 필터링하고, 필요한 값을 계산하고, 여러 데이터를 올바른 관계로 병합하고, 그룹별로 집계한 뒤 결과가 맞는지 검증하는 과정이다.

---

# 오늘의 최종 핵심 정리

```text
Series
→ 컬럼 하나

DataFrame
→ 여러 행과 열을 가진 표

KeyError
→ 실제 컬럼명부터 확인

Boolean Mask
→ 조건에 맞는 행 선택

&
→ AND

|
→ OR

~
→ NOT

isin()
→ 여러 값 중 하나 선택

sort_values()
→ 정렬

copy()
→ 작업용 복사본

line_total
→ quantity × unit_price

value_counts()
→ 값의 빈도

groupby()
→ 결과의 그룹 단위 설정

agg()
→ 그룹별 여러 지표 계산

count()
→ 결측치 제외 개수

size()
→ 전체 행 개수

nunique()
→ 고유 값 개수

merge()
→ DataFrame 연결

many_to_one
→ 왼쪽 여러 행이 오른쪽 한 행과 연결

validate
→ merge 관계 검증

indicator
→ 병합 성공 여부 확인

completed
→ 이번 실습에서 매출에 포함할 주문 상태

order_id.nunique()
→ 실제 주문 수

customer_id.nunique()
→ 실제 고객 수

quantity.sum()
→ 판매 수량

line_total.sum()
→ 금액 합계

to_csv()
→ 분석 결과 저장
```

---

# 마무리

오늘은 pandas의 개별 문법을 사용하는 것뿐만 아니라 실제 데이터 분석 과정 전체를 경험했다.

특히 중요한 점은 다음 세 가지였다.

첫째, 분석을 시작하기 전에 데이터의 구조와 실제 컬럼명을 반드시 확인해야 한다.

둘째, 여러 데이터를 `merge()`할 때는 단순히 연결하는 것에서 끝나는 것이 아니라 `one_to_one`, `many_to_one` 같은 데이터 관계를 이해하고 `validate`와 `indicator`를 이용해 병합이 정상적으로 이루어졌는지 검증해야 한다.

셋째, `groupby()`와 `agg()`를 사용할 때는 결과에서 한 행이 무엇을 의미하는지 명확하게 이해해야 한다.

앞으로 pandas를 사용할 때도 단순히 코드가 실행되는지만 확인하지 않고 다음과 같은 기준으로 분석 결과를 검증하는 습관을 가져야겠다.

```text
실제 컬럼이 맞는가?

필터 조건이 맞는가?

분석 범위가 맞는가?

계산식이 맞는가?

병합 관계가 맞는가?

병합 후 행 수가 정상인가?

미매칭 데이터가 없는가?

집계 단위가 질문과 맞는가?

원본 합계와 결과 합계가 일치하는가?
```

이를 통해 단순히 pandas 코드를 작성하는 것을 넘어, 데이터를 올바르게 이해하고 분석 결과의 신뢰성을 검증하는 방법을 학습했다.